# Production RAG

**Track:** Portfolio. **Level:** Practitioner. **CPU:** yes. **Keys:** not required.

## Goals

- Chunk a document at a topic boundary instead of a fixed character width.
- Retrieve with sparse BM25 and dense hashed TF-IDF, then fuse the rankings.
- Rerank a short candidate list.
- Store the same vectors in three backends: Chroma, Qdrant (in memory), and a SQLite store whose schema matches pgvector.
- Answer with a citation, and fail an automated check when the citation does not support the claim.
- Run one Hebrew query and one English query against a mixed corpus.

## Sources and license

This notebook is original. The syllabus follows the retrieval section of the Full Stack LLM bootcamp and the hybrid-search design used by LlamaIndex and LangChain. Those codebases are Apache-2.0. Their notebooks are not copied. BM25 follows Robertson and Zaragoza. Reciprocal rank fusion follows Cormack, Clarke, and Buettcher. The citation check is a lexical baseline, not a claim of textual entailment.

## Concepts

A production retriever is a pipeline with independent failure modes. Chunking decides what a hit can prove. The sparse index matches words. The dense index matches the vector you actually built, which here is a TF-IDF vector so the lab runs offline. A neural encoder such as multilingual E5 replaces that vector in a deployed system. It does not replace the evaluation. Fusion and reranking only help if a metric says so. A citation check is the smallest grounding test: every claim in the answer has to overlap a cited chunk.


In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", 'chromadb', 'qdrant-client', 'scikit-learn', 'numpy', 'pandas'])
print("packages ready")


In [ ]:
import os
import sys
from pathlib import Path

def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for cand in [start, *start.parents]:
        if (cand / "labs" / "_shared" / "llm.py").exists():
            return cand
    if "google.colab" in sys.modules:
        root = Path("/content/ShaiLearnsAI")
        marker = root / "labs" / "_shared" / "llm.py"
        if not marker.exists():
            os.system(
                "git clone --depth 1 https://github.com/shai200/ShaiLearnsAI.git /content/ShaiLearnsAI"
            )
        return root
    raise RuntimeError(
        "Clone https://github.com/shai200/ShaiLearnsAI and run this notebook from that directory."
    )

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")
print("repo", ROOT)

import math
import re
import sqlite3
from collections import Counter

import numpy as np
import pandas as pd

from labs._shared.seed import seed_everything

seed_everything(0)
TOKEN = re.compile(r"[\w\u0590-\u05FF]+", re.UNICODE)

def tokenize(text: str) -> list[str]:
    return TOKEN.findall(text.lower())


## Guided exercise: corpus, BM25, dense vectors

The corpus is original, including the Hebrew lines. One question is written so that it shares no token with the relevant passage until a glossary expands it. Rankings ignore non-positive scores, so a tie among zeros is not counted as a hit.


In [ ]:
PASSAGES = [
    {"id": "en-savings", "lang": "en", "topic": "savings", "text": "The savings account yields 3 percent annual interest and has no monthly maintenance fee. The minimum opening balance is 100."},
    {"id": "he-savings", "lang": "he", "topic": "savings", "text": "חשבון החיסכון מניב ריבית שנתית של 3 אחוזים ואין בו דמי ניהול חודשיים. יתרת הפתיחה המינימלית היא 100."},
    {"id": "en-card", "lang": "en", "topic": "card", "text": "The credit card has an annual fee of 49 dollars and a foreign currency transaction fee of 2 percent."},
    {"id": "he-card", "lang": "he", "topic": "card", "text": "בכרטיס האשראי יש עמלה שנתית של 49 דולר ועמלת מטבע חוץ של 2 אחוזים."},
    {"id": "en-loan", "lang": "en", "topic": "loan", "text": "Personal loans have a fixed interest rate of 7 percent and a term of 36 months. There is no early repayment fee."},
    {"id": "en-wire", "lang": "en", "topic": "wire", "text": "Domestic transfers between accounts at the same bank are free. International wire transfers cost 15 dollars."},
    {"id": "en-deposit", "lang": "en", "topic": "deposit", "text": "A 12 month term deposit yields 4 percent and cannot be withdrawn early without a penalty."},
]

GLOSSARY = {
    "ריבית": "interest",
    "חשבון": "account",
    "החיסכון": "savings",
    "חיסכון": "savings",
    "עמלה": "fee",
    "השנתית": "annual",
    "שנתית": "annual",
    "כרטיס": "card",
    "האשראי": "credit",
    "אשראי": "credit",
    "yield": "interest",
    "yearly": "annual",
    "apr": "interest",
    "cash": "savings",
    "product": "account",
}

def expand_query(query: str, glossary: dict[str, str] | None = None) -> str:
    glossary = GLOSSARY if glossary is None else glossary
    extra = [glossary[tok] for tok in tokenize(query) if tok in glossary]
    if not extra:
        return query
    return query + " " + " ".join(extra)

class BM25:
    def __init__(self, ids: list[str], texts: list[str], k1: float = 1.5, b: float = 0.75):
        self.ids = ids
        self.k1 = k1
        self.b = b
        self.docs = [tokenize(text) for text in texts]
        self.df = Counter()
        self.tf = []
        self.lengths = []
        for tokens in self.docs:
            counts = Counter(tokens)
            self.tf.append(counts)
            self.lengths.append(len(tokens) or 1)
            for term in counts:
                self.df[term] += 1
        self.avgdl = sum(self.lengths) / len(self.docs)

    def idf(self, term: str) -> float:
        n = self.df.get(term, 0)
        return math.log(1.0 + (len(self.docs) - n + 0.5) / (n + 0.5))

    def score_one(self, query_tokens: list[str], index: int) -> float:
        score = 0.0
        length = self.lengths[index]
        counts = self.tf[index]
        for term in query_tokens:
            freq = counts.get(term, 0)
            if freq == 0:
                continue
            denom = freq + self.k1 * (1.0 - self.b + self.b * length / self.avgdl)
            score += self.idf(term) * freq * (self.k1 + 1.0) / denom
        return score

    def rank(self, query: str, k: int = 5) -> list[tuple[str, float]]:
        tokens = tokenize(query)
        scored = []
        for index, doc_id in enumerate(self.ids):
            score = self.score_one(tokens, index)
            if score > 0:
                scored.append((doc_id, score))
        scored.sort(key=lambda item: -item[1])
        return scored[:k]

class DenseIndex:
    """L2-normalized TF-IDF. A neural encoder plugs in at `embed`."""

    def fit(self, ids: list[str], texts: list[str]) -> None:
        self.ids = ids
        tokenized = [tokenize(text) for text in texts]
        df = Counter()
        for doc in tokenized:
            for term in set(doc):
                df[term] += 1
        self.idf = {term: math.log((1 + len(texts)) / (1 + df[term])) + 1.0 for term in df}
        self.vocab = {term: i for i, term in enumerate(sorted(self.idf))}
        self.matrix = self.embed(texts)

    def embed(self, texts: list[str]) -> np.ndarray:
        matrix = np.zeros((len(texts), len(self.vocab)), dtype=np.float64)
        for row, text in enumerate(texts):
            for term, count in Counter(tokenize(text)).items():
                col = self.vocab.get(term)
                if col is not None:
                    matrix[row, col] = count * self.idf[term]
            norm = np.linalg.norm(matrix[row])
            if norm > 0:
                matrix[row] /= norm
        return matrix

    def rank(self, query: str, k: int = 5) -> list[tuple[str, float]]:
        vector = self.embed([query])[0]
        if np.linalg.norm(vector) == 0:
            return []
        scores = self.matrix @ vector
        order = np.argsort(-scores)
        ranked = []
        for index in order:
            if scores[index] <= 0:
                break
            ranked.append((self.ids[index], float(scores[index])))
            if len(ranked) == k:
                break
        return ranked

ids = [row["id"] for row in PASSAGES]
texts = [row["text"] for row in PASSAGES]
bm25 = BM25(ids, texts)
dense = DenseIndex()
dense.fit(ids, texts)
print("vocab", len(dense.vocab), "bm25 top", bm25.rank("savings interest", k=2))


## Guided exercise: semantic chunking

Adjacent sentences are split where their dense cosine is the minimum along the document. On this paragraph the valley falls between the savings sentences and the card sentences. A fixed character window is not required to do that, and the comparison is the point of the later ablation.


In [ ]:
TOPIC_DOC = [
    "The savings account yields annual interest.",
    "The savings account has no monthly fee.",
    "The credit card charges an annual fee.",
    "The credit card adds a foreign transaction charge.",
]

def adjacent_cosines(sentences: list[str], index: DenseIndex) -> list[float]:
    # Fit a local index so the four sentences define the space.
    local = DenseIndex()
    local.fit([str(i) for i in range(len(sentences))], sentences)
    vectors = local.matrix
    return [float(vectors[i] @ vectors[i - 1]) for i in range(1, len(vectors))]

def semantic_chunks(sentences: list[str]) -> list[str]:
    sims = adjacent_cosines(sentences, dense)
    cut = int(np.argmin(sims)) + 1
    groups = [sentences[:cut], sentences[cut:]]
    return [" ".join(group) for group in groups if group]

chunks = semantic_chunks(TOPIC_DOC)
print("chunks:")
for chunk in chunks:
    print(" -", chunk)
assert "savings" in chunks[0] and "credit" not in chunks[0]
assert "credit" in chunks[1] and "savings" not in chunks[1]


## Guided exercise: three vector stores

The numpy store is the reference. Chroma and Qdrant run in process, with no server. The SQLite store keeps one row per chunk and a float blob. The SQL printed under it is the pgvector statement a Postgres deployment would run instead of the Python cosine loop. All four must return the same nearest id for a passage queried against itself.


In [ ]:
class NumpyStore:
    def __init__(self, ids, vectors):
        self.ids = list(ids)
        self.vectors = np.vstack(vectors)

    def query(self, vector, k=3):
        q = np.asarray(vector, dtype=np.float64)
        q = q / (np.linalg.norm(q) + 1e-12)
        scores = self.vectors @ q
        order = np.argsort(-scores)[:k]
        return [(self.ids[i], float(scores[i])) for i in order]

class PgvectorLikeStore:
    """SQLite stand-in. `postgres_sql` is what pgvector would run."""

    def __init__(self, dim: int):
        self.conn = sqlite3.connect(":memory:")
        self.conn.execute(
            "CREATE TABLE documents (id INTEGER PRIMARY KEY, doc_id TEXT, document TEXT, embedding BLOB)"
        )
        self.postgres_sql = (
            "CREATE EXTENSION IF NOT EXISTS vector;\n"
            "CREATE TABLE documents (\n"
            "  id bigint PRIMARY KEY,\n"
            "  doc_id text NOT NULL,\n"
            "  document text NOT NULL,\n"
            f"  embedding vector({dim})\n"
            ");\n"
            "SELECT doc_id FROM documents ORDER BY embedding <=> $1 LIMIT 5;"
        )

    def upsert(self, rows):
        payload = []
        for i, row in enumerate(rows):
            blob = np.asarray(row["vector"], dtype=np.float32).tobytes()
            payload.append((i, row["doc_id"], row["text"], blob))
        self.conn.executemany("INSERT INTO documents VALUES (?, ?, ?, ?)", payload)

    def query(self, vector, k=3):
        q = np.asarray(vector, dtype=np.float32)
        qn = np.linalg.norm(q) + 1e-12
        scored = []
        for doc_id, blob in self.conn.execute("SELECT doc_id, embedding FROM documents"):
            vec = np.frombuffer(blob, dtype=np.float32)
            sim = float(np.dot(q, vec) / (qn * (np.linalg.norm(vec) + 1e-12)))
            scored.append((doc_id, sim))
        scored.sort(key=lambda item: -item[1])
        return scored[:k]

def chroma_store(ids, texts, vectors):
    import chromadb
    from chromadb.config import Settings
    client = chromadb.Client(Settings(anonymized_telemetry=False))
    collection = client.get_or_create_collection("portfolio_rag", metadata={"hnsw:space": "cosine"})
    collection.add(
        ids=list(ids),
        documents=list(texts),
        embeddings=[vec.tolist() for vec in vectors],
        metadatas=[{"i": i} for i in range(len(ids))],
    )

    def query(vector, k=3):
        result = collection.query(query_embeddings=[np.asarray(vector).tolist()], n_results=k)
        out = []
        for doc_id, distance in zip(result["ids"][0], result["distances"][0]):
            out.append((doc_id, 1.0 - float(distance)))
        return out

    return query

def qdrant_store(ids, vectors):
    from qdrant_client import QdrantClient
    from qdrant_client.models import Distance, PointStruct, VectorParams
    client = QdrantClient(":memory:")
    client.create_collection(
        collection_name="portfolio_rag",
        vectors_config=VectorParams(size=len(vectors[0]), distance=Distance.COSINE),
    )
    points = []
    for i, (doc_id, vector) in enumerate(zip(ids, vectors)):
        points.append(PointStruct(id=i, vector=np.asarray(vector, dtype=float).tolist(), payload={"doc_id": doc_id}))
    client.upsert(collection_name="portfolio_rag", points=points)

    def query(vector, k=3):
        if hasattr(client, "query_points"):
            found = client.query_points(
                collection_name="portfolio_rag",
                query=np.asarray(vector, dtype=float).tolist(),
                limit=k,
            ).points
        else:
            found = client.search(
                collection_name="portfolio_rag",
                query_vector=np.asarray(vector, dtype=float).tolist(),
                limit=k,
            )
        return [(hit.payload["doc_id"], float(hit.score)) for hit in found]

    return query

rows = [{"doc_id": doc_id, "text": text, "vector": vec} for doc_id, text, vec in zip(ids, texts, dense.matrix)]
numpy_store = NumpyStore(ids, dense.matrix)
pg_store = PgvectorLikeStore(dense.matrix.shape[1])
pg_store.upsert(rows)
chroma_query = chroma_store(ids, texts, dense.matrix)
qdrant_query = qdrant_store(ids, dense.matrix)
probe = dense.matrix[0]
tops = {
    "numpy": numpy_store.query(probe, k=1)[0][0],
    "pgvector_like": pg_store.query(probe, k=1)[0][0],
    "chroma": chroma_query(probe, k=1)[0][0],
    "qdrant": qdrant_query(probe, k=1)[0][0],
}
print(tops)
print(pg_store.postgres_sql)
assert set(tops.values()) == {ids[0]}


## Guided exercise: hybrid search, reranking, bilingual retrieval

Reciprocal rank fusion adds `1 / (k + rank)`. It ignores the raw score scales. The reranker is a logistic regression on three features: BM25, dense cosine, and query-token coverage. It is fit on four questions and reported on two held-out questions. That is a wiring check, not a leaderboard number.

The Hebrew query matches the Hebrew passage directly. The same query against an English-only index matches only after the glossary adds the English lemmas. The synonym question (`yearly yield` for `annual interest`) is the English version of that gap.


In [ ]:
def rrf(rankings: list[list[str]], k: int = 60) -> list[str]:
    scores: dict[str, float] = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank + 1)
    return sorted(scores, key=lambda doc_id: -scores[doc_id])

def coverage(query: str, text: str) -> float:
    q = set(tokenize(query))
    if not q:
        return 0.0
    return len(q & set(tokenize(text))) / len(q)

by_id = {row["id"]: row["text"] for row in PASSAGES}

QUESTIONS = [
    {"q": "What is the annual interest on the savings account?", "gold": {"en-savings", "he-savings"}},
    {"q": "What is the annual fee on the credit card?", "gold": {"en-card", "he-card"}},
    {"q": "What does an international wire transfer cost?", "gold": {"en-wire"}},
    {"q": "What is the fixed interest rate on personal loans?", "gold": {"en-loan"}},
    {"q": "מה הריבית על חשבון החיסכון?", "gold": {"he-savings", "en-savings"}},
    {"q": "מה העמלה השנתית על כרטיס האשראי?", "gold": {"he-card", "en-card"}},
]

def retrieve(query: str, glossary: dict | None = GLOSSARY, k: int = 4):
    expanded = expand_query(query, glossary)
    sparse = bm25.rank(expanded, k=k)
    neural = dense.rank(expanded, k=k)
    fused = rrf([[doc_id for doc_id, _ in sparse], [doc_id for doc_id, _ in neural]])
    return {"expanded": expanded, "bm25": sparse, "dense": neural, "fused": fused}

def hit_at(ranked: list[str], gold: set[str], k: int) -> int:
    return int(any(doc_id in gold for doc_id in ranked[:k]))

def mrr(ranked: list[str], gold: set[str]) -> float:
    for rank, doc_id in enumerate(ranked, start=1):
        if doc_id in gold:
            return 1.0 / rank
    return 0.0

records = []
for item in QUESTIONS:
    found = retrieve(item["q"])
    records.append({
        "query": item["q"],
        "bm25_hit": hit_at([doc for doc, _ in found["bm25"]], item["gold"], 3),
        "dense_hit": hit_at([doc for doc, _ in found["dense"]], item["gold"], 3),
        "hybrid_hit": hit_at(found["fused"], item["gold"], 3),
        "hybrid_mrr": mrr(found["fused"], item["gold"]),
        "top": found["fused"][0] if found["fused"] else None,
    })
print(pd.DataFrame(records)[["query", "bm25_hit", "dense_hit", "hybrid_hit", "hybrid_mrr", "top"]].to_string(index=False))

# Synonym gap: no shared token until expansion.
raw_gap = bm25.rank("yearly yield cash product", k=3)
expanded_gap = bm25.rank(expand_query("yearly yield cash product"), k=3)
print("raw synonym hits", raw_gap)
print("expanded synonym hits", expanded_gap)
assert raw_gap == []
assert expanded_gap and expanded_gap[0][0] in {"en-savings", "he-savings"}

# Hebrew against the Hebrew passage, then against English only.
he_query = "מה הריבית על חשבון החיסכון?"
he_hits = [doc for doc, _ in bm25.rank(he_query, k=3)]
assert "he-savings" in he_hits
en_only = BM25(
    [row["id"] for row in PASSAGES if row["lang"] == "en"],
    [row["text"] for row in PASSAGES if row["lang"] == "en"],
)
assert en_only.rank(he_query, k=3) == []
assert en_only.rank(expand_query(he_query), k=1)[0][0] == "en-savings"

# Reranker. Candidates are the hybrid top-4. Fit on the first four questions.
train_x, train_y, test_packs = [], [], []
for n, item in enumerate(QUESTIONS):
    found = retrieve(item["q"])
    for doc_id in found["fused"][:4]:
        sparse_score = dict(found["bm25"]).get(doc_id, 0.0)
        dense_score = dict(found["dense"]).get(doc_id, 0.0)
        features = [sparse_score, dense_score, coverage(found["expanded"], by_id[doc_id])]
        label = int(doc_id in item["gold"])
        if n < 4:
            train_x.append(features)
            train_y.append(label)
        else:
            test_packs.append((item["q"], doc_id, features, label))

from sklearn.linear_model import LogisticRegression
reranker = LogisticRegression(max_iter=200)
reranker.fit(np.array(train_x), np.array(train_y))
print("reranker coef bm25, dense, coverage", reranker.coef_.round(3).tolist())
heldout_queries = sorted({row[0] for row in test_packs})
assert heldout_queries


## Guided exercise: cited answers and the verification check

The offline reader quotes the top fused passage and names its id. The checker drops a short stop list and requires the remaining claim tokens to overlap the cited text. A supported quote passes. A number that never appears in the cited passage fails. A true paraphrase also fails. That last failure is the limitation of a lexical check, and it is the reason a later lab would add an entailment model.


In [ ]:
STOP = {
    "the", "a", "an", "of", "and", "to", "is", "in", "for", "no", "has", "with", "on", "what",
    "does", "do", "a", "between", "there",
}

def citation_supported(claim: str, cited_text: str, minimum: float = 0.34) -> bool:
    # A number in the claim has to appear in the cited text. Overlap alone
    # accepts "9 percent" when the source says "3 percent".
    if set(re.findall(r"\d+", claim)) - set(re.findall(r"\d+", cited_text)):
        return False
    claim_terms = set(tokenize(claim)) - STOP
    if not claim_terms:
        return False
    source_terms = set(tokenize(cited_text)) - STOP
    return len(claim_terms & source_terms) / len(claim_terms) >= minimum

def extractive_answer(query: str) -> tuple[str, str]:
    found = retrieve(query)
    if not found["fused"]:
        return "I do not know.", ""
    doc_id = found["fused"][0]
    sentence = by_id[doc_id].split(". ")[0].strip(".")
    return f"{sentence}. [{doc_id}]", doc_id

checked = []
for item in QUESTIONS:
    answer, doc_id = extractive_answer(item["q"])
    ok = citation_supported(answer, by_id[doc_id])
    checked.append((item["q"], doc_id, ok, answer))
print(pd.DataFrame(checked, columns=["query", "citation", "supported", "answer"]).to_string(index=False))
assert all(row[2] for row in checked)

assert citation_supported(
    "The savings account yields 3 percent annual interest.",
    by_id["en-savings"],
)
assert not citation_supported(
    "The savings account yields 9 percent annual interest.",
    by_id["en-savings"],
)
# True, but paraphrased. Lexical overlap is below the threshold.
assert not citation_supported(
    "Cash held in the deposit product earns three percent a year.",
    by_id["en-savings"],
)

def faithfulness(query: str) -> bool:
    answer, doc_id = extractive_answer(query)
    if not doc_id:
        return False
    return citation_supported(answer, by_id[doc_id])

faith = [faithfulness(item["q"]) for item in QUESTIONS]
print("extractive faithfulness", sum(faith), "/", len(faith))
assert sum(faith) == len(faith)
print("LAB OK")


## Your turn

Change BM25 `k1` from 1.5 to 0.5 on the savings query and compare the score of `en-savings`. The length normalization term depends on `k1`, so the score should move.

## Solution


In [ ]:
baseline = dict(BM25(ids, texts, k1=1.5).rank("savings account interest", k=5))["en-savings"]
shorter = dict(BM25(ids, texts, k1=0.5).rank("savings account interest", k=5))["en-savings"]
print("k1=1.5", round(baseline, 4), "k1=0.5", round(shorter, 4))
assert baseline != shorter


## Beyond the course

Ablation: Hebrew query, English-only index, glossary on versus off. This is the smallest bilingual experiment that still matches a production failure mode, which is a mixed corpus and a query in the other language. A multilingual encoder is the replacement for the glossary, not a reason to skip the ablation.


In [ ]:
def english_hit(query: str, glossary: dict | None) -> bool:
    expanded = expand_query(query, glossary or {})
    hits = [doc for doc, _ in en_only.rank(expanded, k=3)]
    return "en-savings" in hits

query = "מה הריבית על חשבון החיסכון?"
on = english_hit(query, GLOSSARY)
off = english_hit(query, {})
print("glossary on", on, "glossary off", off)
assert on and not off


## Research questions

1. The paraphrase "earns three percent a year" was true and the lexical check rejected it. What would a false claim look like that this check still accepts?
2. Reciprocal rank fusion throws away score magnitude. When would you rather calibrate the two score distributions and add them?
3. The glossary maps `החיסכון` as a whole word, including the definite article. Where does that break for Hebrew morphology, and which public multilingual encoders are trained to absorb it?

## Further reading

- Robertson and Zaragoza, *The Probabilistic Relevance Framework: BM25 and Beyond*.
- Cormack, Clarke, and Buettcher, *Reciprocal Rank Fusion Outperforms Condorcet and Individual Rank Learning Methods* (2009).
- Lewis et al., *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks* (2020).
- Karpukhin et al., *Dense Passage Retrieval for Open-Domain Question Answering* (2020).
- Nogueira and Cho, *Passage Re-ranking with BERT* (2019).
- Full Stack LLM bootcamp, retrieval labs. LlamaIndex and LangChain hybrid-search docs (Apache-2.0). Not copied here.
